# 02 — ASR Baseline: Whisper on Moroccan Darija

Evaluate OpenAI Whisper (tiny → large-v3) on the Darija test set.
Establishes the WER/CER baseline before fine-tuning.

| Model | Params | Relative speed | VRAM |
|---|---|---|---|
| tiny | 39M | 32× | <1 GB |
| base | 74M | 16× | <1 GB |
| small | 244M | 6× | ~2 GB |
| medium | 769M | 2× | ~5 GB |
| large-v3 | 1550M | 1× | ~10 GB |

In [ ]:
import os, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import librosa
from faster_whisper import WhisperModel
from jiwer import wer, cer, process_words
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = Path(os.getenv('DATA_ROOT', '../data/samples'))
RESULTS   = Path('../data/samples/baseline_results')
RESULTS.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Load test set (audio paths + reference transcripts)
TEST_MANIFEST = DATA_ROOT / 'test_manifest.jsonl'

# Create synthetic manifest if not present
if not TEST_MANIFEST.exists():
    print('No test manifest found — using synthetic data for demo')
    test_samples = [
        {'audio_path': 'sample_001.wav', 'reference': 'واش عندك شي حاجة أرخص من هذي',
         'dialect': 'casablanci', 'domain': 'e-commerce', 'duration_s': 4.2},
        {'audio_path': 'sample_002.wav', 'reference': 'بغيت نحجز شي سيارة ليوم كامل',
         'dialect': 'marrakchi', 'domain': 'tourism',    'duration_s': 3.8},
        {'audio_path': 'sample_003.wav', 'reference': 'عندي ألم فالرأس من الصبح',
         'dialect': 'fassi',     'domain': 'healthcare', 'duration_s': 5.1},
    ]
else:
    test_samples = []
    with open(TEST_MANIFEST) as f:
        for line in f:
            test_samples.append(json.loads(line))

print(f'Test samples: {len(test_samples)}')
pd.DataFrame(test_samples).head()

In [ ]:
def transcribe_batch(model_name: str,
                     samples: list[dict],
                     language: str = 'ar',
                     compute_type: str = 'float16') -> pd.DataFrame:
    """Transcribe a batch of audio files with a Whisper model."""
    compute_type = 'int8' if DEVICE == 'cpu' else compute_type

    print(f'\nLoading {model_name} ({compute_type}) on {DEVICE}...')
    model = WhisperModel(model_name, device=DEVICE, compute_type=compute_type)

    results = []
    for sample in tqdm(samples, desc=f'Transcribing [{model_name}]'):
        audio_path = DATA_ROOT / sample['audio_path']

        if not audio_path.exists():
            # Use random audio for demo
            duration  = sample.get('duration_s', 5)
            dummy_wav = DATA_ROOT / 'dummy.wav'
            dummy_wav.parent.mkdir(exist_ok=True)
            import soundfile as sf
            sf.write(str(dummy_wav), np.random.randn(int(16000 * duration)) * 0.01, 16000)
            audio_path = dummy_wav

        t0 = time.perf_counter()
        segments, info = model.transcribe(
            str(audio_path),
            language=language,
            beam_size=5,
            vad_filter=True,
        )
        hypothesis = ' '.join(s.text.strip() for s in segments)
        elapsed    = time.perf_counter() - t0

        ref = sample.get('reference', '')
        results.append({
            'model':         model_name,
            'audio_path':    sample['audio_path'],
            'dialect':       sample.get('dialect', 'unknown'),
            'domain':        sample.get('domain',  'unknown'),
            'duration_s':    sample.get('duration_s', 0),
            'reference':     ref,
            'hypothesis':    hypothesis,
            'wer':           wer(ref, hypothesis) if ref else None,
            'cer':           cer(ref, hypothesis) if ref else None,
            'rtf':           elapsed / max(sample.get('duration_s', 1), 0.1),
            'elapsed_s':     round(elapsed, 3),
            'detected_lang': info.language,
            'lang_prob':     round(info.language_probability, 3),
        })

    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    return pd.DataFrame(results)

print('transcribe_batch() defined.')

In [ ]:
# Evaluate models (skip large-v3 unless GPU available with >10GB VRAM)
MODELS_TO_EVAL = ['tiny', 'base', 'small']
if DEVICE == 'cuda' and torch.cuda.get_device_properties(0).total_memory > 5e9:
    MODELS_TO_EVAL.append('medium')
if DEVICE == 'cuda' and torch.cuda.get_device_properties(0).total_memory > 10e9:
    MODELS_TO_EVAL.append('large-v3')

print(f'Evaluating models: {MODELS_TO_EVAL}')

all_results = []
for model_name in MODELS_TO_EVAL:
    df_model = transcribe_batch(model_name, test_samples)
    all_results.append(df_model)

df_results = pd.concat(all_results, ignore_index=True)

# Save
df_results.to_parquet(RESULTS / 'baseline_results.parquet', index=False)
print(f'\nSaved results: {RESULTS / "baseline_results.parquet"}')

In [ ]:
# Summarise results per model
summary = df_results.groupby('model').agg(
    avg_wer=('wer', 'mean'),
    avg_cer=('cer', 'mean'),
    avg_rtf=('rtf', 'mean'),
    lang_detection_rate=('detected_lang', lambda x: (x == 'ar').mean()),
    n_samples=('audio_path', 'count'),
).round(3)

print('\n=== Baseline Results ===')
print(summary.to_string())

# Plot WER vs RTF tradeoff
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

models = summary.index.tolist()
wer_vals = summary['avg_wer'] * 100
rtf_vals = summary['avg_rtf']

ax1.bar(models, wer_vals, color='#4e79a7')
ax1.set_title('WER by Model (lower = better)')
ax1.set_ylabel('WER (%)')
ax1.set_xlabel('Whisper Model')

ax2.scatter(rtf_vals, wer_vals, s=150, c='#e15759', zorder=5)
for i, m in enumerate(models):
    ax2.annotate(m, (rtf_vals.iloc[i], wer_vals.iloc[i]),
                 textcoords='offset points', xytext=(8, 4))
ax2.set_xlabel('Real-Time Factor (RTF) — lower = faster')
ax2.set_ylabel('WER (%)')
ax2.set_title('Quality vs Speed Tradeoff')
ax2.axhline(20, color='orange', linestyle='--', label='Target WER 20%')
ax2.legend()

plt.tight_layout()
plt.savefig(RESULTS / 'baseline_wer_rtf.png', dpi=150, bbox_inches='tight')
plt.show()